# Parts 2–3 · Run interacting melanoma agents

RNA initializes programs; movement, contact recognition, growth and geometry are declared assumptions. Use the recognition-off control. The engine uses transparent local rules and makes no Jev/LLM calls.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

# In Colab, the public GitHub version must already contain this release.
INSTALL_PACKAGE = True
REPO_URL = "https://github.com/hylixinsights/VirtualTissue-scRNAseq.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((d for d in candidates if (d / "pyproject.toml").exists() and (d / "src/vtsc").exists()), None)
if ROOT is None:
    ROOT = Path("/content/VirtualTissue-scRNAseq") if IN_COLAB else Path.home() / "VirtualTissue-scRNAseq"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
if INSTALL_PACKAGE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebooks]"], check=True)
sys.path.insert(0, str(ROOT / "src"))
print("Code:", ROOT)
# Leave WORK below local, or explicitly mount Drive and choose a Drive path.
WORK = Path(os.environ.get("VTSC_NOTEBOOK_WORK", str(ROOT / "outputs/notebook_melanoma")))
WORK.mkdir(parents=True, exist_ok=True)
print("Files:", WORK)


In [ ]:
from vtsc.tissue_inputs import read_tissue_bundle
from vtsc.tissue_results import run_tissue
from vtsc.tissue import DEFAULT
BUNDLE = WORK / 'input.zip'
payload = read_tissue_bundle(BUNDLE)
HOURS = 24
SEED = 21
LAYOUT = 'tumor-nests'  # or mixed
CONFIG_PATH = None  # optional YAML written with `vtsc tissue-template`
print('Initial composition:', payload['provenance']['sampled_composition'])
DEFAULT


## Explicit recognition scenario and negative control

Declared recognition permits all eligible cytotoxic agents to attack melanoma-labelled neighbors in contact. It is not inferred antigen specificity or a drug response. Both runs start with the same cells and geometry; trajectories subsequently diverge.

In [ ]:
active = run_tissue(BUNDLE, WORK / 'declared-recognition', scenario='declared-recognition',
                    hours=HOURS, seed=SEED, layout=LAYOUT, config_path=CONFIG_PATH)
control = run_tissue(BUNDLE, WORK / 'recognition-off', scenario='recognition-off',
                     hours=HOURS, seed=SEED, layout=LAYOUT, config_path=CONFIG_PATH)
active, control


In [ ]:
from vtsc.results import read_run
import pandas as pd
runs = {name: read_run(WORK / name) for name in ['declared-recognition', 'recognition-off']}
assert runs['declared-recognition']['frames'][0] == runs['recognition-off']['frames'][0]
assert not any(e['event'] in ['attack', 'death'] for e in runs['recognition-off']['events'])
display(pd.DataFrame([{'scenario': name, **r['summary']} for name,r in runs.items()]))


Use another WORK/output path for another seed or layout. These comparisons check implementation and sensitivity, not clinical efficacy. Continue with Part 4 to inspect the recorded decisions.